# 1. Getting started: reading, CRS, and geometric features

GEOAI_3D is geospatial-first: every cloud carries a coordinate reference system (CRS) and a provenance record, and every operation keeps them attached. This notebook reads a small real LiDAR tile, inspects it, reprojects it, and computes the eigenvalue-based geometric features that the later notebooks build on.

**Install** (base plus the LAZ and viewer extras):

```bash
pip install "geoai3d[laz,viz]"
```

The sample is a ~150 m crop of Dutch national LiDAR (AHN4, CC0); see `../data/README.md`.

In [ ]:
from geoai3d import read_lidar

cloud = read_lidar("../data/ahn_sample.laz")
print(f"{len(cloud):,} points")
print("CRS:", cloud.crs.to_epsg())
print("bounds:", tuple(round(b, 1) for b in cloud.bounds))
print("attributes:", cloud.attribute_names)

The file came with a CRS in its header (EPSG:28992, the Dutch RD New grid) and per-point attributes including the ASPRS `classification` labels. Let's see the class mix.

In [ ]:
import numpy as np

labels, counts = np.unique(cloud.attribute("classification"), return_counts=True)
for label, count in zip(labels, counts):
    print(f"class {int(label):>2}: {100 * count / len(cloud):5.1f}%")
# AHN: 2 = ground, 6 = building, 1 = unclassified/vegetation, 9 = water

## Reprojection

Because the CRS travels with the cloud, reprojecting is one call. Here we go from RD New to WGS84 longitude/latitude just to see the bounds change; the notebooks otherwise stay in the metric RD New grid.

In [ ]:
from geoai3d import reproject

wgs84 = reproject(cloud, 4326)
print("WGS84 bounds:", tuple(round(b, 6) for b in wgs84.bounds))

## Geometric features

`geometric_features` fits a small neighbourhood around each point and derives eigenvalue descriptors — planarity, linearity, sphericity, verticality, and more. A 1 m radius captures a few tens of neighbours at this density. These describe local shape and drive the segmentation and classification later.

In [ ]:
from geoai3d import geometric_features

features = geometric_features(cloud, radius=1.0)
print("added:", [n for n in features.attribute_names if n not in cloud.attribute_names])

# Ground should be far more planar than vegetation.
classes = features.attribute("classification")
planarity = features.attribute("planarity")
print("median planarity, ground  :", round(float(np.nanmedian(planarity[classes == 2])), 3))
print("median planarity, building:", round(float(np.nanmedian(planarity[classes == 6])), 3))
print("median planarity, veg     :", round(float(np.nanmedian(planarity[classes == 1])), 3))

## A quick look

`view` renders an interactive 3D plot (needs the `viz` extra). Colour by any attribute — here, planarity.

In [ ]:
from geoai3d import view

view(features, color_by="planarity")

## Save the result

Parquet is the package's lossless at-rest format: it stores every column and keeps the CRS and provenance in the file metadata.

In [ ]:
from geoai3d import to_parquet

to_parquet(features, "features.parquet")
print("saved features.parquet")

You now have a georeferenced cloud with geometric features and a lineage record. Notebook 2 turns it into terrain products.